In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler

df = pd.read_csv('/Users/xiaozhuzhang/Desktop/阿里数分实习/week1/UserBehavior.csv.part',header=None)
df.columns = ['user_id', 'item_id', 'category_id','behavior_type', 'timestamp']
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s',errors='cojerce')
df = df[(df['datetime'] >= 'xxxx-xx-xx') & (df['datetime'] <= 'xxxx-xx-xx')]
df['hour'] = df['datetime'].dt.hour

behavior_stats = df.groupby(['item_id', 'behavior_type']).size().unstack().fillna(0)
behavior_stats.columns = ['pv', 'fav', 'cart', 'buy']
print(behavior_stats)

top20 = {
    'buy': behavior_stats.nlargest(20, 'buy')[['buy']],
    'pv': behavior_stats.nlargest(20, 'pv')[['pv']],
    'fav': behavior_stats.nlargest(20, 'fav')[['fav']],
    'cart': behavior_stats.nlargest(20, 'cart')[['cart']]
}

print(top20)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.suptitle('Top 20 Products by Behavior Type', fontsize=16, y=1.02)

for (behavior, ax) in zip(['pv', 'fav', 'cart', 'buy'], axes.flatten()):
    data = behavior_stats.nlargest(20, behavior)[[behavior]]
    data.plot(kind='bar', ax=ax, color='skyblue',
             width=0.8, alpha=0.8, legend=False)
    for p in ax.patches:
        ax.annotate(f"{int(p.get_height())}",
                   (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha='center', va='center', xytext=(0, 5),
                   textcoords='offset points')
    ax.set_title(f'Top 20 by {behavior.upper()}', pad=10)
    ax.set_xlabel('Product ID')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# 销量Top20商品类别
buy_top20_items = behavior_stats.nlargest(20, 'buy').index

# 统计这些商品的类目分布
buy_category_dist = df[df['item_id'].isin(buy_top20_items)] \
    .groupby('category_id') \
    .size() \
    .nlargest(10) \
    .rename('Top 20 Selling Items')

print(f"==== 商品类别购买量排行榜 ====\n{buy_category_dist}")

# 用户 RF score

now = df['datetime'].max()
rf = df[df['behavior_type'] == 'buy'].groupby('user_id').agg(
    Recency=('datetime', lambda x: (now - x.max()).days),
    Frequency=('user_id', 'count')
)

# 计算标准分
scaler = RobustScaler()
rf[['R_Scaled', 'F_Scaled']] = scaler.fit_transform(rf[['Recency', 'Frequency']])
rf['R_Score'] = -rf['R_Scaled']  # 越近越高分
rf['RF_Score'] = rf['R_Scaled'] + rf['F_Scaled']

# 生成排名（得分越高价值越高）
rf['Rank'] = rf['RF_Score'].rank(method='min', ascending=False).astype(int)

# 输出纯净排名表
print("==== RF得分排名分布 ====")
print(f"基准时间: {now.strftime('%Y-%m-%d')}")
print(f"有效用户数: {len(rf)}")
print("\n得分统计:")
print(rf['RF_Score'].describe())

print("\nTop 50用户排名:")
result = rf.sort_values('RF_Score', ascending=False).head(50)[
    ['Recency', 'Frequency', 'RF_Score', 'Rank']
]
print(result.to_string(float_format="%.2f"))